# GeoTrade — Research Analysis Notebook

This notebook explores all pipeline outputs end-to-end.

### Prerequisites
Run these from the project root before opening this notebook:
```bash
python scripts/step1_ingest.py
python scripts/step2_nlp.py
python scripts/step3_score.py
python scripts/step4_model.py
```

### Sections
1. Raw article statistics
2. NLP event label distribution
3. Daily tension signals timeline
4. Country heatmap
5. Tension vs VIX correlation
6. Model performance comparison

In [ ]:
import sys, os
# Add project root so config/ and pipeline/ resolve correctly
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from config.settings import settings
from pipeline.utils.db import get_db

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'monospace', 'figure.facecolor': '#050b14'})
sns.set_palette('husl')

%matplotlib inline
print('✓ Setup complete')
print(f'  MongoDB: {settings.MONGODB_URI[:30]}...')
print(f'  Database: {settings.MONGODB_DB}')

## 1 — Raw Article Statistics

In [ ]:
db = get_db()
articles = pd.DataFrame(list(db[settings.COL_RAW_ARTICLES].find({}, {'_id': 0, 'hash': 0})))

print(f'Total articles   : {len(articles)}')
print(f'Processed        : {articles["processed"].sum()}')
print(f'Pending          : {(~articles["processed"]).sum()}')

# Source breakdown
articles['source_type'] = articles['source'].str.split(':').str[0]
print('\nBy source type:')
print(articles['source_type'].value_counts().to_string())

## 2 — NLP Event Label Distribution

In [ ]:
events = pd.DataFrame(list(db[settings.COL_PROCESSED_EVENTS].find({}, {'_id': 0})))
print(f'Processed events: {len(events)}')

COLORS = ['#ef4444', '#38bdf8', '#f59e0b', '#22c55e', '#a855f7']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Event label bar chart
counts = events['event_label'].value_counts()
axes[0].bar(counts.index, counts.values, color=COLORS[:len(counts)])
axes[0].set_title('Event Label Distribution', fontsize=12)
axes[0].set_ylabel('Count')

# Negative sentiment histogram
axes[1].hist(events['neg_sentiment_score'], bins=25, color='#38bdf8', alpha=0.8, edgecolor='none')
axes[1].axvline(events['neg_sentiment_score'].mean(), color='#ef4444', ls='--', lw=1.5,
                label=f'Mean = {events["neg_sentiment_score"].mean():.2f}')
axes[1].set_title('Negative Sentiment Score Distribution', fontsize=12)
axes[1].set_xlabel('Score (0 = positive, 1 = negative)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3 — Daily Tension Signals Timeline

In [ ]:
sigs = pd.DataFrame(list(db[settings.COL_DAILY_SIGNALS].find({}, {'_id': 0})))
sigs['date'] = pd.to_datetime(sigs['date'])

# Global aggregate per day
daily = sigs.groupby('date').agg(
    global_tension    = ('tension_score', 'mean'),
    max_tension       = ('tension_score', 'max'),
    total_events      = ('event_count',   'sum'),
    n_countries       = ('iso',           'nunique'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.fill_between(daily['date'], daily['global_tension'], alpha=0.2, color='#38bdf8')
ax.plot(daily['date'], daily['global_tension'], color='#38bdf8', lw=2, label='Global Mean')
ax.plot(daily['date'], daily['max_tension'], color='#ef4444', lw=1, ls='--', label='Max')
ax.axhline(0.65, color='#ef4444', lw=0.8, ls=':', alpha=0.5)
ax.axhline(0.35, color='#f59e0b', lw=0.8, ls=':', alpha=0.5)
ax.set_ylabel('Tension Score')
ax.set_title('Daily Global Tension', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(daily['date'], daily['n_countries'], color='#a855f7', alpha=0.7, width=0.8)
ax.set_ylabel('Countries Affected')
ax.set_title('Countries with Active Signals per Day', fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nTop 10 highest-tension entries:')
top = sigs.nlargest(10, 'tension_score')[['date','country','tension_score','tension_label','top_event_label']]
print(top.to_string(index=False))

## 4 — Country Tension Heatmap

In [ ]:
pivot = sigs.pivot_table(index='country', columns='date', values='tension_score', aggfunc='mean')
top15 = pivot.mean(axis=1).nlargest(15).index
pivot_top = pivot.loc[top15]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(
    pivot_top, ax=ax,
    cmap='RdYlGn_r', vmin=0, vmax=1,
    linewidths=0.3, linecolor='#0d1627',
    cbar_kws={'label': 'Tension Score'},
)
ax.set_title('Tension Heatmap — Top 15 Countries by Mean Score', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 5 — Tension vs VIX Correlation

In [ ]:
merged_path = '../data/processed/merged_dataset.csv'
if not os.path.exists(merged_path):
    print(f'Not found: {merged_path}  — run step4_model.py first')
else:
    merged = pd.read_csv(merged_path, parse_dates=['date'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: tension vs VIX coloured by label
    ax = axes[0]
    sc = ax.scatter(
        merged['global_tension'], merged['vix_close'],
        c=merged['volatility_increase'], cmap='RdYlGn_r',
        alpha=0.6, s=35, edgecolors='none',
    )
    plt.colorbar(sc, ax=ax, label='Volatility Increase')
    ax.set_xlabel('Global Tension Score')
    ax.set_ylabel('VIX Close')
    ax.set_title('Tension vs VIX', fontsize=12)
    ax.grid(True, alpha=0.3)

    # Correlation matrix
    ax = axes[1]
    corr_cols = [c for c in [
        'global_tension','max_tension','conflict_count',
        'avg_neg_sentiment','vix_close','sp500_volatility_5d',
    ] if c in merged.columns]
    sns.heatmap(merged[corr_cols].corr(), ax=ax, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True, linewidths=0.4)
    ax.set_title('Feature Correlation Matrix', fontsize=12)

    plt.tight_layout()
    plt.show()

    r = merged['global_tension'].corr(merged['vix_close'])
    print(f'Pearson r (tension vs VIX): {r:.4f}')

## 6 — View Saved Model Plots

In [ ]:
from IPython.display import Image, display
import glob

plot_files = sorted(glob.glob('../data/plots/*.png'))
if not plot_files:
    print('No plots found — run step4_model.py first')
else:
    for path in plot_files:
        print(f'\n── {os.path.basename(path)} ──')
        display(Image(path))